In [ ]:
from pathlib import Path

from conversational_toolkit.embeddings.openai import OpenAIEmbeddings
from sme_kt_zh_collaboration_rag.feature0_baseline_rag import (
    load_chunks,
    build_vector_store,
)
from conversational_toolkit.embeddings.qwen_vl import Qwen3VLEmbeddings
from sme_kt_zh_collaboration_rag.feature0_baseline_rag import EMBEDDING_MODEL
from loguru import logger
from dotenv import load_dotenv

In [ ]:
load_dotenv("../../.env.local")

_DB_DIR = Path("../src/sme_kt_zh_collaboration_rag/db")

IMAGE_VS_PATH = _DB_DIR / "vs_image"
TEXT_VS_PATH = _DB_DIR / "vs_text"

logger.info("Initializing vector stores...")
all_chunks = load_chunks()

In [ ]:
text_chunks = [c for c in all_chunks if c.mime_type.startswith("text/")]
# limit content size to 8k tokens for OpenAI embedding model
text_chunks = [c for c in text_chunks if len(c.content) < 8 * 1024]
await build_vector_store(
    text_chunks,
    OpenAIEmbeddings(model_name=EMBEDDING_MODEL),
    db_path=TEXT_VS_PATH,
    reset=True,
)

In [ ]:
image_chunks = [c for c in all_chunks if c.mime_type.startswith("image/")]
await build_vector_store(
    image_chunks,
    Qwen3VLEmbeddings(),
    db_path=IMAGE_VS_PATH,
    reset=True,
    batch_size=1,
)